In [1]:
import pandas as pd
import os

# Create output directory if it doesn't exist
os.makedirs("data/exported", exist_ok=True)

In [ ]:
# Import data [WILL NEED TO CHANGE BASED ON FORMATS FROM DATA GATHERING NOTEBOOK]
asylum_by_month = pd.read_csv("data/raw/asylum_cases_philadelphia.csv")
asylum_by_month['month'] = pd.to_datetime(asylum_by_month['fymon'])

# asylum_by_language = pd.read_excel("data/asylum.xlsx", sheet_name="Asylum by Language")
# asylum_by_language.columns = asylum_by_language.columns.str.lower().str.replace(" ", "_")

# asylum_by_custody = pd.read_excel("data/asylum.xlsx", sheet_name="Asylum by Custody")
# asylum_by_custody.columns = asylum_by_custody.columns.str.lower().str.replace(" ", "_")

In [6]:
asylum_by_month

,fymon,all_cases,city,denied_cases,all_represented,all_not_represented,represented_denied,not_represented_denied,month
0,2000-10,68,Philadelphia,32,48,20,15,17,2000-10-01
1,2000-11,46,Philadelphia,27,24,22,12,15,2000-11-01
2,2000-12,56,Philadelphia,31,29,27,14,17,2000-12-01
3,2001-01,59,Philadelphia,40,37,22,25,15,2001-01-01
4,2001-02,42,Philadelphia,26,31,11,18,8,2001-02-01
...,...,...,...,...,...,...,...,...,...
268,2025-08,144,Philadelphia,98,132,12,90,8,2025-08-01
269,2025-09,251,Philadelphia,165,228,23,149,16,2025-09-01
270,2025-10,298,Philadelphia,226,265,33,200,26,2025-10-01
271,2025-11,233,Philadelphia,177,215,18,162,15,2025-11-01


In [ ]:
# Helper function to summarize data on denied cases
def summarise(df, group_col=None):
    def agg(d):
        phl_total  = d['number'].sum()
        phl_denied = d['denied_number'].sum()
        us_total   = d['united_states_total'].sum()
        us_denied  = d['united_states_denied'].sum() + d['united_states_other_relief_granted'].sum()
        return pd.Series({
            'phl_total_cases': phl_total,
            'phl_total_denied': phl_denied,
            'phl_pct_denied': phl_denied / phl_total if phl_total else None,
            'us_total_cases': us_total,
            'us_total_denied': us_denied,
            'us_pct_denied': us_denied / us_total if us_total else None,
        })
    if group_col:
        return df.groupby(group_col).apply(agg).reset_index()
    return agg(df).to_frame().T

In [ ]:
# Q1: Overall denial rates
asylum_total = summarise(asylum_by_month)
print(asylum_total)
asylum_total.to_csv("data/exported/asylum_total.csv", index=False)

In [ ]:
# Q2: By presidency
def assign_president(month):
    if pd.Timestamp("2001-01-20") < month <= pd.Timestamp("2009-01-20"):
        return "Bush"
    elif pd.Timestamp("2009-01-20") < month <= pd.Timestamp("2017-01-20"):
        return "Obama"
    elif pd.Timestamp("2017-01-20") < month <= pd.Timestamp("2021-01-20"):
        return "Trump I"
    elif pd.Timestamp("2021-01-20") < month <= pd.Timestamp("2025-01-20"):
        return "Biden"
    elif month > pd.Timestamp("2025-01-20"):
        return "Trump II"
    return "Other"

order = ["Bush", "Obama", "Trump I", "Biden", "Trump II"]
asylum_by_month['president'] = asylum_by_month['month'].apply(assign_president)
asylum_by_president = summarise(asylum_by_month, "president")
asylum_by_president['president'] = pd.Categorical(asylum_by_president['president'], categories=order, ordered=True)
asylum_by_president = asylum_by_president.sort_values('president')
print(asylum_by_president)
asylum_by_president.to_csv("data/exported/asylum_by_president.csv", index=False)

In [ ]:
# Q3: Biden's last months vs. Trump II
recent = asylum_by_month[asylum_by_month['month'] >= pd.Timestamp("2024-07-01")].copy()
recent['president'] = recent['month'].apply(
    lambda m: "Biden" if m <= pd.Timestamp("2025-01-20") else ("Trump" if m >= pd.Timestamp("2025-02-01") else "Other")
)
asylum_biden_vs_trump = summarise(recent, "president")
asylum_biden_vs_trump['president'] = pd.Categorical(asylum_biden_vs_trump['president'], categories=["Biden", "Trump"], ordered=True)
asylum_biden_vs_trump = asylum_biden_vs_trump.sort_values('president')
print(asylum_biden_vs_trump)
asylum_biden_vs_trump.to_csv("data/exported/asylum_biden_vs_trump.csv", index=False)

In [ ]:
# Q4: By month (recent)
monthly = asylum_by_month[asylum_by_month['month'] >= pd.Timestamp("2024-07-01")].copy()
monthly['month'] = monthly['month'].dt.strftime('%Y-%m')
asylum_by_month_recent = summarise(monthly, "month")
print(asylum_by_month_recent)
asylum_by_month_recent.to_csv("data/exported/asylum_by_month_recent.csv", index=False)

In [ ]:
# # Q5: By custody status
# asylum_by_custody = summarise(asylum_by_custody, "custody")
# print(asylum_by_custody)
# asylum_by_custody.to_csv("data/exported/asylum_by_custody.csv", index=False)

In [ ]:
# # Q6: By language
# asylum_by_language = summarise(asylum_by_language, "language")
# print(asylum_by_language)
# asylum_by_language.to_csv("data/exported/asylum_by_language.csv", index=False)

In [ ]:
# # Q7: By legal representation
# asylum_by_representation = summarise(asylum_by_representation, "representation")
# print(asylum_by_representation)
# asylum_by_representation.to_csv("data/exported/asylum_by_representation.csv", index=False)